In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

# Fetch

In [ ]:
import kagglehub

In [ ]:
temp_dir = Path.cwd() / ".temp"

if temp_dir.parent.name != "model":
    raise RuntimeError()

temp_dir.mkdir(exist_ok=True)
temp_dir

In [ ]:
dataset_path = (
    Path(
        kagglehub.dataset_download(
            "metricasecuador/handwritten-digits-version-1-hwd-v1",
            output_dir=str(temp_dir / "dataset"),
        )
    )
    / "HWD-V1"
)
dataset_path

# PyTorch

In [ ]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Dataset

In [ ]:
RESOLUTION = 32

In [ ]:
def dataset_all_split(
    dataset: dict[int, list[Path]], split: float, random_state: int | None = None
):
    """
    Return two split dicts from DATASET_ALL
    """
    train, test = {}, {}

    rng = np.random.default_rng(random_state)

    for digit, items in dataset.items():
        shuffled_items = list(items)
        rng.shuffle(shuffled_items)

        split_idx = int(len(shuffled_items) * split)
        train[digit] = shuffled_items[:split_idx]
        test[digit] = shuffled_items[split_idx:]

    return train, test


def dataset_dict_to_list(dataset_dict):
    return [(item, digit) for digit, items in dataset_dict.items() for item in items]

In [ ]:
DATASET_ITEMS_ALL = {
    digit: list(dataset_path.glob(f"HWD-V1-Standard/{digit}/*.png"))
    for digit in range(10)
}

DATASET_ITEMS_TRAIN, DATASET_ITEMS_TEST = dataset_all_split(
    DATASET_ITEMS_ALL, 0.7, random_state=99
)
DATASET_ITEMS_ALL_LIST, DATASET_ITEMS_TRAIN_LIST, DATASET_ITEMS_TEST_LIST = (
    dataset_dict_to_list(DATASET_ITEMS_ALL),
    dataset_dict_to_list(DATASET_ITEMS_TRAIN),
    dataset_dict_to_list(DATASET_ITEMS_TEST),
)

len(DATASET_ITEMS_TRAIN_LIST), len(DATASET_ITEMS_TEST_LIST)

In [ ]:
class DigitDataset(Dataset):
    def __init__(self, *, items: list[tuple[Path, int]], r: int):
        self.r = r
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        item_path, label = self.items[idx]

        img = Image.open(item_path).convert("L")
        img = ImageOps.invert(img)

        img = img.resize((self.r, self.r))
        img.save(temp_dir / "test_img.png")

        data = np.array(img.get_flattened_data(), dtype=np.float32) / 255
        tensor = torch.tensor(data, dtype=torch.float32)
        tensor = tensor.view(1, self.r, self.r)

        return tensor, label


ds = DigitDataset(items=DATASET_ITEMS_ALL_LIST, r=RESOLUTION)
ds.__getitem__(99999)

In [ ]:
def show_dataset_entry(dataset, index):
    """
    tensor: torch.Tensor of shape (1, H, W) with values in [0, 1]
    """
    tensor, label = dataset.__getitem__(index)

    img = tensor.squeeze(0).cpu().numpy()  # -> (H, W)

    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.show()

In [ ]:
show_dataset_entry(DigitDataset(items=DATASET_ITEMS_ALL_LIST, r=RESOLUTION), 0)

# Model

In [ ]:
@dataclass(kw_only=True)
class TrainResult:
    dataset: Dataset
    model: nn.Module
    epoch: int
    loss: float
    accuracy: float

In [ ]:
def train(
    *, dataset: DigitDataset, model: nn.Module, epochs: int, echo=True
) -> TrainResult:
    loader = DataLoader[DigitDataset](dataset, batch_size=1024, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model = model.to(device)

    for epoch in range(epochs):
        total_loss = 0
        correct = 0

        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            Y_pred: torch.Tensor = model(X_batch)
            loss = loss_fn(Y_pred, Y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            correct += (Y_pred.argmax(1) == Y_batch).sum().item()

        n = len(dataset)
        calc_loss = total_loss / n
        calc_accuracy = correct / n

        if echo:
            print(
                f"Epoch {epoch + 1:2d}  loss={total_loss / n:.4f}  acc={correct / n:.2%}"
            )

    return TrainResult(
        dataset=dataset,
        model=model,
        epoch=epoch,
        loss=calc_loss,
        accuracy=calc_accuracy,
    )

### Very Simple Linear Model

In [ ]:
class VerySimpleLinearModel(nn.Module):
    def __init__(self, *, r: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),  # (1, R, R) -> RxR
            nn.Linear(r * r, 10),  # RxR -> 10 digit scores
            nn.Softmax(),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
dataset_train = DigitDataset(items=DATASET_ITEMS_TRAIN_LIST, r=RESOLUTION)

result = train(
    dataset=dataset_train, model=VerySimpleLinearModel(r=RESOLUTION), epochs=60
)

result


### Second Model

In [ ]:
class SecondModel(nn.Module):
    def __init__(self, *, r: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(
                1, 8, kernel_size=3, stride=1, padding=1
            ),  # (1, R, R) -> (8, R, R)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (8, R, R) -> (8, R/2, R/2)
            nn.Flatten(),  # (8, R/2, R/2) -> (8 * R/2 * R/2)
            nn.Linear(8 * (r // 2) * (r // 2), 100),  # (8 * R/2 * R/2) -> 100
            nn.ReLU(),
            nn.Linear(100, 10),  # (8 * R/2 * R/2) -> 10 digit scores
            nn.Softmax(),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
result2 = train(dataset=dataset_train, model=SecondModel(r=RESOLUTION), epochs=60)
result2